# 阿里移动推荐

## 数据集
- 用户对商品的交互数据，包含用户ID、商品ID、商品类别、用户位置、交互时间
- 商品的特征数据，包含商品ID、商品类别

## 要求
推测用户对商品的购买行为，输出“用户-商品”二列数据表

## 建模
- 数据预处理：标记缺失值，特征编码，时间特征处理
- 模型选择：随机森林RF，针对每个用户训练一个模型
- 模型训练：特征为用户位置、商品类别、时间特征、缺失标记，标签为用户的购买行为（是否购买）
- 模型评估：使用测试数据评估模型的性能，常用的评估指标包括准确率、召回率、F1分数等
- 结果输出：根据模型的预测结果，输出用户-商品二列数据表，表示用户对商品的购买行为预测结果

In [2]:
# 库和全局变量

import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import joblib

feat_order = ['item_category', 'user_geohash', 'geohash_loss', 'time']


In [1]:
# 数据预处理

# 读取文件
interaction_data = pd.read_csv('data/tianchi_fresh_comp_train_user.csv')
item_data = pd.read_csv('data/tianchi_fresh_comp_train_item.csv')

# 标记缺失值
interaction_data["geohash_loss"] = interaction_data["user_geohash"].isnull().astype(int)
item_data["geohash_loss"] = item_data["item_geohash"].isnull().astype(int)

# 特征编码
le = LabelEncoder()
interaction_data["user_geohash"] = le.fit_transform(interaction_data["user_geohash"].astype(str))
item_data["item_geohash"] = le.fit_transform(item_data["item_geohash"].astype(str))
interaction_data["time"] = pd.to_datetime(interaction_data["time"]).astype(int)

# 保存预训练数据
interaction_data.to_csv('data/train_data_washed.csv', index=False)
item_data.to_csv('data/test_data_washed.csv', index=False)

In [3]:
# 模型训练

# 加载数据
train_data = pd.read_csv('data/train_data_washed.csv')

# 针对单个用户的模型
def train_user_model(user_id):
    # 准备训练数据
    user_data = train_data[train_data["user_id"] == user_id]
    train_X = user_data.drop(columns=['user_id', 'item_id', 'behavior_type'])
    train_X = train_X.reindex(columns=feat_order, fill_value=0)
    train_y = user_data['behavior_type']
    train_y = (train_y == 4).astype(int)

    # 训练
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(train_X, train_y)

    return model

# 分别训练并保存
for user_id in interaction_data['user_id'].unique():
    model = train_user_model(user_id)
    joblib.dump(model, f'model/model_{user_id}.joblib')


In [1]:
# 结果输出

# 加载模型和数据
test_data = pd.read_csv('data/test_data_washed.csv')
train_data = pd.read_csv('data/train_data_washed.csv')

# 单个用户结果预测
def predict_user_model(user_id,model):
    # 准备测试数据
    test_X = test_data.drop(columns=['item_id'])
    test_X['user_geohash'] = test_X['item_geohash']
    test_X = test_X.drop(columns=['item_geohash'])
    test_X['time'] = pd.to_datetime('2014-12-19 00').value
    test_X = test_X.reindex(columns=feat_order, fill_value=0)
    # 预测用户对每个商品的行为
    act = model.predict(test_X)
    # act = model.predict(test_X)
    # 整合行为数据
    result = pd.DataFrame({'item_id':test_data['item_id'],'act':act})
    result['user_id'] = user_id
    return result

# 分别预测并保存
result = pd.DataFrame(columns=['user_id','item_id'])
for user_id in train_data['user_id'].unique():
    model = joblib.load(f'model/model_{user_id}.joblib')
    user_to_item = predict_user_model(user_id=user_id,model=model)

    result.loc[len(result)] = user_to_item[user_to_item['act'] == 1]['user_id','item_id']

result.to_csv('prediction.csv', index=False)


NameError: name 'pd' is not defined